In [2]:
import pandas as pd


import sqlite3

In [3]:
# Connect to the database
DB_PATH = "../../data/private/out/cs61a/fa25/snapshots.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [4]:
students_to_include = set()

In [5]:
# only include students with 50 or more backups
students_backup_counts = cursor.execute(
    "SELECT COUNT(*), student_email FROM backup_metadata GROUP BY student_email HAVING COUNT(*) >= 50"
).fetchall()
students_backup_counts[:10]

[(110, '0007feaa'),
 (74, '00111d17'),
 (141, '0061065c'),
 (66, '006cfad8'),
 (150, '00b2d716'),
 (133, '01a6eee6'),
 (82, '022a0298'),
 (75, '02517dfc'),
 (55, '027420a8'),
 (85, '02bcf50a')]

In [6]:
for count, email in students_backup_counts:
    students_to_include.add(email)

In [7]:
len(students_to_include)

966

In [8]:
# only include students with lint errors besides F401

# Number of lint errors per student on their final backup, excluding F401
cursor.execute("""
WITH last_backups AS (
    SELECT
        backup_id,
        MAX(created) AS created, -- in SQLite this works
        student_email,
        file_contents_location
    FROM backup_metadata
    GROUP BY student_email
),

backups_with_lint_errors AS (
    SELECT
        lb.backup_id,
        lb.created,
        lb.student_email,
        le.*
    FROM last_backups AS lb
    JOIN lint_errors AS le
    ON REPLACE(lb.file_contents_location, '../../data/private/', '') = REPLACE(le.file_contents_location, '/ants.py', '')
    WHERE le.code != 'F401'
)

SELECT
    student_email,
    COUNT(*)
FROM backups_with_lint_errors
GROUP BY student_email
""")
student_to_lint_error_count = cursor.fetchall()
student_to_lint_error_count[:10]

[('00111d17', 7),
 ('0061065c', 2),
 ('006cfad8', 5),
 ('00b2d716', 3),
 ('01a6eee6', 5),
 ('027420a8', 1),
 ('02ca5d9f', 5),
 ('02dd96ae', 1),
 ('0369a673', 1),
 ('038c33f4', 3)]

In [9]:
# students_to_include = students_to_include & set([email for email, count in student_to_lint_error_count])
students_to_include = set([email for email, count in student_to_lint_error_count])

len(students_to_include)

639

In [10]:
# print statements


def count_search_string_in_file(path, search_string):
    with open(path, "r") as f:
        contents = f.read()
        return contents.count(search_string)

In [11]:
cursor.execute("""
SELECT backup_id, student_email, created, file_contents_location || '/ants.py' AS file_contents_location
FROM backup_metadata
""")
file_paths = cursor.fetchall()
file_paths[0]

('rXvllk',
 '373a81b1',
 '2025-10-22T06:50:45',
 '../../data/private/cal/cs61a/fa25/ants/373a81b1/rXvllk/ants.py')

In [12]:
print_debug_data = {
    # TODO find out why there is a path that is None
    "backup_id": [row[0] for row in file_paths if row[3]],
    "student_email": [row[1] for row in file_paths if row[3]],
    "created": [row[2] for row in file_paths if row[3]],
    "num_occurrences_print": [],
}

for path in [row[3] for row in file_paths]:
    if path:  # TODO find out why there is a path that is None
        print_debug_data["num_occurrences_print"].append(
            count_search_string_in_file(path, "print")
        )

In [13]:
print_debug_data_df = pd.DataFrame(print_debug_data)
print_debug_data_df["created"] = pd.to_datetime(print_debug_data_df["created"])
print_debug_data_df.head()

,backup_id,student_email,created,num_occurrences_print
0,rXvllk,373a81b1,2025-10-22 06:50:45,3
1,AA2Yl1,373a81b1,2025-10-22 06:50:10,3
2,Xn09XW,373a81b1,2025-10-22 06:45:56,3
3,rXvl7K,373a81b1,2025-10-22 06:45:25,3
4,lDm96r,373a81b1,2025-10-22 06:45:20,3


In [14]:
students_with_high_prints = []
for student_email, num_occurrences_print in zip(
    print_debug_data["student_email"], print_debug_data["num_occurrences_print"]
):
    if num_occurrences_print > 3:
        students_with_high_prints.append([student_email, num_occurrences_print])
students_with_high_prints.sort(key=lambda pair: pair[1], reverse=True)
students_with_high_prints[:10]

[['b32641dd', 23],
 ['b32641dd', 23],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22],
 ['b32641dd', 22]]

In [15]:
students_to_include = students_to_include & set(
    [email for email, count in students_with_high_prints]
)
len(students_to_include)

179

In [16]:
import random

random.seed(42)

final_list = random.sample(sorted(students_to_include), k=25)
final_list

['e3384165',
 '1faf1492',
 '0757b4af',
 '5e0b5dff',
 '55d9e0b2',
 '4349b29d',
 '27f16a00',
 '1bcf17a8',
 'bbb281e4',
 '18e36d10',
 'd0d1b4b0',
 '94d2cb91',
 '09e6bcbc',
 '08a08a79',
 '1a3aee97',
 '3ff28b43',
 '4972bef4',
 'a8faf137',
 'd6797b5b',
 'fc1888f1',
 'c2b307c8',
 '395b6a1a',
 'f0cd1289',
 '90cfed97',
 '41a86dbb']